In [35]:
import pandas as pd
import numpy as np
import plotly.express as px
import streamlit as st

In [36]:
df = pd.read_csv(r'C:\Users\nayer\Downloads\python course\final project stuff\retail_store_sales.csv')
df

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
...,...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,2023-09-03,NaN
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,2022-08-12,False
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,2024-08-24,NaN
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,2023-12-30,True


In [37]:
df_clean = df.copy()

# Check Data Types 

In [38]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [39]:
df_clean = df_clean.drop(columns=["Customer ID"])

In [40]:
df_clean["Transaction Date"] = pd.to_datetime(df_clean["Transaction Date"])

In [41]:
df_clean = df_clean.sort_values(
    "Transaction Date", ascending=True
).reset_index(drop=True)

In [42]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    12575 non-null  object        
 1   Category          12575 non-null  object        
 2   Item              11362 non-null  object        
 3   Price Per Unit    11966 non-null  float64       
 4   Quantity          11971 non-null  float64       
 5   Total Spent       11971 non-null  float64       
 6   Payment Method    12575 non-null  object        
 7   Location          12575 non-null  object        
 8   Transaction Date  12575 non-null  datetime64[ns]
 9   Discount Applied  8376 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(6)
memory usage: 982.5+ KB


# Missing Values

In [43]:
df_clean.isna().sum()

Transaction ID         0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

In [44]:
df_clean.loc[
    df_clean["Price Per Unit"].isna(),
    "Price Per Unit"
] = (
    df_clean["Total Spent"] / df_clean["Quantity"]
)

In [45]:
df_clean["Price Per Unit"].isna().sum()

0

In [46]:
item_counts = (
    df_clean.dropna(subset=["Category", "Price Per Unit", "Item"])
    .groupby(["Category", "Price Per Unit"])["Item"]
    .nunique()
)

print(item_counts.value_counts())

1    200
Name: Item, dtype: int64


In [47]:
for i in df_clean.index:
    if pd.isna(df_clean.loc[i, "Item"]):
        category = df_clean.loc[i, "Category"]
        price = df_clean.loc[i, "Price Per Unit"]

        match = df_clean[
            (df_clean["Category"] == category) &
            (df_clean["Price Per Unit"] == price) &
            (df_clean["Item"].notna())
        ]

        if len(match) > 0:
            df_clean.loc[i, "Item"] = match["Item"].iloc[0]
        

In [48]:
df_clean["Item"].isna().sum()

0

In [49]:
df_clean.loc[
    df_clean["Quantity"].isna(),
    "Quantity"
] = (
    df_clean["Total Spent"] / df_clean["Price Per Unit"]
)

In [50]:
df_clean["Quantity"].isna().sum()

604

In [51]:
df_clean.loc[
    df_clean["Quantity"].isna(),
    ["Total Spent", "Price Per Unit"]
].isna().sum()

Total Spent       604
Price Per Unit      0
dtype: int64

In [52]:
df_clean = df_clean.dropna(
    subset=["Quantity", "Total Spent"]
).copy()

In [53]:
df_clean['Discount Applied'].value_counts(normalize= True)*100

True     50.344482
False    49.655518
Name: Discount Applied, dtype: float64

In [54]:
df_clean["Discount Applied"] = df_clean["Discount Applied"].fillna("Unknown")

In [55]:
df_clean["Discount Applied"].value_counts()

True       4019
Unknown    3988
False      3964
Name: Discount Applied, dtype: int64

In [56]:
df_clean["Quantity"] = df_clean["Quantity"].astype(int)

In [57]:
df_clean.isna().sum()

Transaction ID      0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64

# Check Duplicates

In [58]:
df_clean.duplicated().sum()

0

# Check Outliers

In [59]:
numerical_col= df_clean.select_dtypes(include = 'number').columns
numerical_col

Index(['Price Per Unit', 'Quantity', 'Total Spent'], dtype='object')

In [60]:
for col in numerical_col:
    px.box(df_clean,x= col,color_discrete_sequence=["#3B82F6"]).show()

# Check Inconsistency

In [61]:
df_clean.describe(include = 'number').round(2)

,Price Per Unit,Quantity,Total Spent
count,11971.00,11971.00,11971.00
mean,23.36,5.54,129.65
std,10.74,2.86,94.75
min,5.00,1.00,5.00
25%,14.00,3.00,51.00
50%,23.00,6.00,108.50
75%,33.50,8.00,192.00
max,41.00,10.00,410.00


In [62]:
df_clean.describe(include = 'object')

,Transaction ID,Category,Item,Payment Method,Location,Discount Applied
count,11971,11971,11971,11971,11971,11971
unique,11971,8,200,3,2,3
top,TXN_7021336,Furniture,Item_2_BEV,Cash,Online,True
freq,1,1525,128,4103,6068,4019


# Feature Engineering

In [63]:
print("Minimum date:", df_clean["Transaction Date"].min())
print("Maximum date:", df_clean["Transaction Date"].max())

Minimum date: 2022-01-01 00:00:00
Maximum date: 2025-01-18 00:00:00


In [64]:
df_clean["Year-Month"] = df_clean["Transaction Date"].dt.to_period("M")
df_clean["Year-Month"]

0        2022-01
1        2022-01
2        2022-01
3        2022-01
4        2022-01
          ...   
12570    2025-01
12571    2025-01
12572    2025-01
12573    2025-01
12574    2025-01
Name: Year-Month, Length: 11971, dtype: period[M]

In [65]:
df_clean["Month-Year"] = df_clean["Transaction Date"].dt.strftime("%B %Y")

# Plots

UNIVARIATE ANALYSIS

1. How are transaction values distributed?

In [66]:
 px.histogram(df_clean, x="Total Spent",nbins= 30,title="Distribution of Transaction Values",labels={"Total Spent": "Transaction Value"},color_discrete_sequence=["#3B82F6"])

2. Which product categories appear most frequently?

In [67]:
px.pie(df_clean, names="Category", title="Number of Transactions by Category", color_discrete_sequence=[
        "#1D4ED8", "#3B82F6", "#60A5FA", "#93C5FD","#2563EB", "#1E40AF", "#BFDBFE", "#DBEAFE" ])

3. Which items appear most frequently?

In [68]:
item_counts = df_clean["Item"].value_counts().head(10).reset_index()
item_counts.columns = ["Item", "Transactions"]

px.bar(item_counts, x="Transactions", y="Item", orientation="h", title="Top 10 Most Frequently Purchased Items",
    labels={"Transactions": "Number of Transactions", "Item": "Item"},
    text_auto=True,
    color_discrete_sequence=["#3B82F6"])


4. Which payment methods are used most often?

In [69]:
payment_counts = (df_clean["Payment Method"].value_counts().reset_index())

payment_counts.columns = ["Payment Method", "Transactions"]

px.bar( payment_counts, x="Payment Method", y="Transactions",title="Transactions by Payment Method",
    labels={"Payment Method": "Payment Method","count": "Number of Transactions"},text_auto=True,color_discrete_sequence=["#3B82F6"])

5. How many items are purchased per transaction

In [70]:
quantity_counts = df_clean["Quantity"].value_counts().sort_index().reset_index()
quantity_counts.columns = ["Quantity", "Transactions"]

px.bar(quantity_counts,x="Quantity", y="Transactions", title="Distribution of Quantity per Transaction",
    labels={"Quantity": "Quantity Purchased", "Transactions": "Number of Transactions"},text_auto=True,color_discrete_sequence=["#3B82F6"])

6. How are transactions distributed across locations?

In [71]:
px.pie(df_clean, names="Location", title="Number of Transactions by Location", color_discrete_sequence=["#1D4ED8", "#3B82F6"])

BIVARIATE ANALYSIS

1. How do sales change over time?

In [72]:
monthly_sales = df_clean.groupby("Year-Month", as_index=False)["Total Spent"].sum()

monthly_sales["Year-Month"] = monthly_sales["Year-Month"].astype(str)

px.line(monthly_sales,x="Year-Month", y="Total Spent",title="Monthly Sales Over Time",
    labels={"Year-Month": "Month-Year","Total Spent": "Total Revenue"},markers=True,
    color_discrete_sequence=["#3B82F6"])


2. Which product categories generate the most revenue?

In [73]:
category_sales = df_clean.groupby("Category", as_index=False)["Total Spent"].sum().sort_values("Total Spent", ascending=False)

px.bar(category_sales,x="Category", y="Total Spent", title="Total Revenue by Product Category",
    labels={"Category": "Product Category", "Total Spent": "Total Revenue" },
    text_auto=True,color_discrete_sequence=["#3B82F6"])


3. Which items sell the most?

In [74]:
item_sales = df_clean.groupby("Item", as_index=False)["Quantity"].sum().sort_values("Quantity", ascending=False).head(10)

px.bar(item_sales,x="Quantity",y="Item",orientation="h",title="Top 10 Best-Selling Items by Quantity",
    labels={"Quantity": "Total Units Sold","Item": "Item"},
    text_auto=True,color_discrete_sequence=["#3B82F6"])

4. Which categories sell the most units?

In [75]:
category_quantity = df_clean.groupby("Category", as_index=False)["Quantity"].sum().sort_values("Quantity", ascending=False)
px.bar(category_quantity,x="Category",y="Quantity", title="Total Units Sold by Category",
    labels={"Category": "Product Category","Quantity": "Total Units Sold"},text_auto=True,color_discrete_sequence=["#3B82F6"])

5. Which payment method generates the most revenue?

In [76]:
payment_sales = df_clean.groupby("Payment Method", as_index=False)["Total Spent"].sum().sort_values("Total Spent", ascending=False)
px.bar(payment_sales,x="Payment Method", y="Total Spent",title="Total Revenue by Payment Method",
    labels={"Payment Method": "Payment Method","Total Spent": "Total Revenue"},
    text_auto=True, color_discrete_sequence=["#3B82F6"])

6. Which store locations perform best?

In [77]:
location_sales = df_clean.groupby("Location", as_index=False)["Total Spent"].sum().sort_values("Total Spent", ascending=False)
px.bar(location_sales,x="Location",y="Total Spent",title="Total Revenue by Location",
    labels={"Location": "Store Location", "Total Spent": "Total Revenue"},
    text_auto=True,color_discrete_sequence=["#3B82F6"])


MULTIVARIATE ANALYSIS

1. Which categories generate the most revenue at each location?

In [78]:
category_location = df_clean.groupby( ["Category", "Location"])["Total Spent"].sum().reset_index()

px.bar(category_location,x="Category",y="Total Spent", color="Location",barmode="group",title="Revenue by Category and Location",
    labels={ "Category": "Product Category", "Location": "Location","Total Spent": "Total Revenue"},
    color_discrete_sequence=[ "#1D4ED8", "#3B82F6", "#60A5FA", "#93C5FD" ])

2.How do payment methods contribute to revenue across locations?

In [79]:
location_payment_revenue = df_clean.groupby(["Location", "Payment Method"])["Total Spent"].sum().reset_index()

px.bar(location_payment_revenue,x="Location", y="Total Spent",color="Payment Method",barmode="group",
    title="Revenue by Payment Method and Location",
    labels={"Location": "Location","Payment Method": "Payment Method","Total Spent": "Total Revenue"},
    color_discrete_sequence=["#1D4ED8", "#3B82F6", "#60A5FA", "#93C5FD", "#BFDBFE"])

# Save Cleaned CSV

In [80]:
df_clean.to_csv(
    r"C:\Users\nayer\Downloads\python course\final project stuff\retail_sales_cleaned.csv",
    index=False
)

In [81]:
import os
os.getcwd()

'C:\\Users\\nayer'

In [82]:
os.chdir(r'C:\Users\nayer\Downloads\python course\final project stuff')
os.getcwd()

'C:\\Users\\nayer\\Downloads\\python course\\final project stuff'

# Deployment

In [83]:
%%writefile Retail_Sales_Analysis.py


import pandas as pd
import numpy as np
import plotly.express as px
import streamlit as st

df_clean = pd.read_csv(
    r"C:\Users\nayer\Downloads\python course\final project stuff\retail_sales_cleaned.csv")
Navigator = st.sidebar.title('🧭Navigator')
page = st.sidebar.radio("Pages", ["Dataset", "Analytics"])

if page == 'Dataset':
    st.title("🛍️Retail Sales Analysis")
    st.subheader("Sales Trends, Product Performance & Purchasing Behavior")
    st.image('https://vidi-corp.com/wp-content/uploads/2025/05/retaildatanalytics.png')
    st.header('Retail Dataset')
    st.dataframe(df_clean)
    
    st.write("### 📋 Column Descriptions")

    cols = {
    "Transaction ID": "A unique identifier for each transaction. Always present and unique.",
    "Category": "The category of the purchased item, such as Food or Furniture.",
    "Item": "The name of the purchased item.",
    "Price Per Unit": "The price of a single unit of the purchased item.",
    "Quantity": "The number of units purchased in a transaction.",
    "Total Spent": "The total amount spent on the transaction, calculated as Quantity multiplied by Price Per Unit.",
    "Payment Method": "The payment method used for the transaction, such as Cash, Credit Card, or Digital Wallet.",
    "Location": "The location where the transaction occurred, such as In-store or Online.",
    "Transaction Date": "The date on which the transaction occurred.",
    "Discount Applied": "Indicates whether a discount was applied.",
    "Year-Month": "An engineered time feature representing the transaction year and month, used for chronological monthly sales analysis.",
    "Month-Year": "An engineered time feature displaying the transaction month and year in a readable format, such as Jan 2023, for chart labels."
     }

    for col, desc in cols.items():
        with st.expander(f" {col}"):
            st.write(desc)

            
            
elif page == 'Analytics':

    st.title("🛍️ Retail Data Analytics")
    
    # Sidebar Filters
    st.sidebar.header("🔎 Filter Your Analysis")

    # Category
    categories = ["All Categories"] + sorted(df_clean["Category"].dropna().unique().tolist())
    selected_category = st.sidebar.selectbox("🛍️ Select Category", categories)

    # Location
    locations = ["All Locations"] + sorted(df_clean["Location"].dropna().unique().tolist())
    selected_location = st.sidebar.selectbox("🌐 Select Location", locations)

    # Payment Method
    payment_methods = ["All Payment Methods"] + sorted(df_clean["Payment Method"].dropna().unique().tolist())
    selected_payment = st.sidebar.selectbox("💳 Select Payment Method", payment_methods)

    # Date Range
    df_clean["Transaction Date"] = pd.to_datetime(df_clean["Transaction Date"])

    min_date = df_clean["Transaction Date"].min().date()
    max_date = df_clean["Transaction Date"].max().date()

    date_range = st.sidebar.date_input(
        "📅 Select Date Range",
        value=(min_date, max_date),
        min_value=min_date,
        max_value=max_date)

    # Apply Filters
    filtered_df = df_clean.copy()

    if selected_category != "All Categories":
        filtered_df = filtered_df[filtered_df["Category"] == selected_category]

    if selected_location != "All Locations":
        filtered_df = filtered_df[filtered_df["Location"] == selected_location]

    if selected_payment != "All Payment Methods":
        filtered_df = filtered_df[filtered_df["Payment Method"] == selected_payment]

    if len(date_range) == 2:
        start_date, end_date = date_range
        filtered_df = filtered_df[(filtered_df["Transaction Date"].dt.date >= start_date)
            & (filtered_df["Transaction Date"].dt.date <= end_date)]

    # Four Metrics
    total_revenue = filtered_df["Total Spent"].sum()
    total_transactions = len(filtered_df)
    units_sold = filtered_df["Quantity"].sum()

    avg_transaction = (
        total_revenue / total_transactions
        if total_transactions > 0 else 0
    )

    col1, col2 = st.columns(2)

    with col1:
        st.metric("💰 Total Revenue", f"{total_revenue:,.0f}")
        st.metric("📦 Units Sold", f"{units_sold:,.0f}")

    with col2:
        st.metric("🧾 Transactions", f"{total_transactions:,}")
        st.metric("🧮 Avg. Transaction", f"{avg_transaction:,.2f}")

    tab1, tab2, tab3 = st.tabs( ["Univariate", "Bivariate", "Multivariate"] )

    
    

    
   

    with tab1:
#Distribution of Transaction Values
        st.plotly_chart(px.histogram(filtered_df, x="Total Spent", nbins= 30,title="Distribution of Transaction Values",
    labels={"Total Spent": "Transaction Value"},color_discrete_sequence=["#3B82F6"]))
        
#Number of Transactions by Category
        st.plotly_chart(px.pie(filtered_df, names="Category", title=" Number of Transactions by Category", color_discrete_sequence=[
        "#1D4ED8", "#3B82F6", "#60A5FA", "#93C5FD", "#2563EB", "#1E40AF", "#BFDBFE", "#DBEAFE"]))
        
#Number of Transactions by Location
        st.plotly_chart(px.pie(filtered_df, names="Location", title=" Number of Transactions by Location", color_discrete_sequence=[
        "#1D4ED8", "#3B82F6"]))
            
            

#Top 10 Most Frequently Purchased Items
        item_counts = (filtered_df["Item"].value_counts().head(10).reset_index())

        item_counts.columns = ["Item", "Transactions"]

        fig = px.bar(item_counts,x="Transactions", y="Item",orientation="h",
    title="Top 10 Most Frequently Purchased Items",
    labels={"count": "Number of Transactions","Item": "Item"},text_auto=True,
    color_discrete_sequence=["#3B82F6"])

        fig.update_layout(yaxis={"categoryorder": "total ascending"})

        st.plotly_chart(fig, use_container_width=True)


      
 #Transactions by Payment Method
        payment_counts = (filtered_df["Payment Method"].value_counts().reset_index())

        payment_counts.columns = ["Payment Method", "Transactions"]

        fig = px.bar( payment_counts,x="Payment Method",y="Transactions",title=" Transactions by Payment Method",
    labels={"Payment Method": "Payment Method","count": "Number of Transactions"},
    text_auto=True,color_discrete_sequence=["#3B82F6"])

        st.plotly_chart(fig, use_container_width=True)

        
 #Distribution of Quantity per Transaction       
        quantity_counts = (filtered_df["Quantity"].value_counts().sort_index().reset_index())

        quantity_counts.columns = ["Quantity", "Transactions"]

        fig = px.bar(quantity_counts, x="Quantity",y="Transactions",title=" Distribution of Quantity per Transaction",
    labels={"Quantity": "Quantity Purchased","Transactions": "Number of Transactions"},
    text_auto=True,color_discrete_sequence=["#3B82F6"])

        st.plotly_chart(fig, use_container_width=True)
        

    
    
    
    
    
    
    with tab2:
        
 #Monthly Sales Over Time       
        monthly_sales = (filtered_df.groupby("Year-Month", as_index=False)["Total Spent"].sum() .sort_values("Year-Month"))

        monthly_sales["Year-Month"] = monthly_sales["Year-Month"].astype(str)

        fig = px.line( monthly_sales,x="Year-Month", y="Total Spent",title=" Monthly Sales Over Time",
    labels={"Year-Month": "Month-Year", "Total Spent": "Total Revenue"},markers=True,color_discrete_sequence=["#3B82F6"])

        st.plotly_chart(fig, use_container_width=True)
        
 #Top 10 Best-Selling Items by Quantity       
        item_sales = (filtered_df.groupby("Item", as_index=False)["Quantity"].sum().sort_values("Quantity", ascending=False).head(10))

        fig = px.bar(item_sales,x="Quantity",y="Item",orientation="h",title=" Top 10 Best-Selling Items by Quantity",
    labels={ "Quantity": "Total Units Sold", "Item": "Item"},
    text_auto=True,color_discrete_sequence=["#3B82F6"])

        fig.update_layout(yaxis={"categoryorder": "total ascending"})

        st.plotly_chart(fig, use_container_width=True)
        


#Total Revenue by Product Category
        category_sales = (filtered_df.groupby("Category", as_index=False)["Total Spent"].sum().sort_values("Total Spent", ascending=False))

        fig = px.bar( category_sales,x="Category",y="Total Spent",title=" Total Revenue by Product Category",
    labels={"Category": "Product Category", "Total Spent": "Total Revenue"},
    text_auto=True,color_discrete_sequence=["#3B82F6"])

        st.plotly_chart(fig, use_container_width=True)

        
 #Total Units Sold by Category       
        category_quantity = (filtered_df.groupby("Category", as_index=False)["Quantity"].sum() .sort_values("Quantity", ascending=False))

        fig = px.bar(category_quantity, x="Category", y="Quantity", title=" Total Units Sold by Category",
    labels={ "Category": "Product Category","Quantity": "Total Units Sold" },
    text_auto=True, color_discrete_sequence=["#3B82F6"])

        st.plotly_chart(fig, use_container_width=True)
        
#Total Revenue by Payment Method
        payment_sales = (filtered_df.groupby("Payment Method", as_index=False)["Total Spent"].sum().sort_values("Total Spent", ascending=False))

        fig = px.bar( payment_sales, x="Payment Method", y="Total Spent", title="Total Revenue by Payment Method",
    labels={ "Payment Method": "Payment Method","Total Spent": "Total Revenue"},
    text_auto=True,color_discrete_sequence=["#3B82F6"])

        st.plotly_chart(fig, use_container_width=True)
    
    
 #Total Revenue by Location   
        location_sales = (filtered_df.groupby("Location", as_index=False)["Total Spent"].sum().sort_values("Total Spent", ascending=False))

        fig = px.bar(location_sales, x="Location", y="Total Spent",title="Total Revenue by Location",
    labels={ "Location": "Store Location", "Total Spent": "Total Revenue"},
    text_auto=True, color_discrete_sequence=["#3B82F6"])

        st.plotly_chart(fig, use_container_width=True)

        
        
        
        
        
        
    with tab3:
        
 #Revenue by Category and Location       
        category_location = (filtered_df.groupby(["Category", "Location"], as_index=False)["Total Spent"].sum())

        fig = px.bar(category_location,x="Category",y="Total Spent",color="Location", barmode="group",title=" Revenue by Category and Location",
    labels={ "Category": "Product Category", "Location": "Location","Total Spent": "Total Revenue"},
    color_discrete_sequence=[ "#1D4ED8", "#3B82F6","#60A5FA", "#93C5FD"])

        st.plotly_chart(fig, use_container_width=True)

        
        
#Revenue by Payment Method and Location
        location_payment_revenue = (filtered_df.groupby( ["Location", "Payment Method"], as_index=False)["Total Spent"].sum())

        fig = px.bar(location_payment_revenue,x="Location", y="Total Spent", color="Payment Method", barmode="group",
    title="Revenue by Payment Method and Location",labels={"Location": "Location","Payment Method": "Payment Method","Total Spent": "Total Revenue"},
    color_discrete_sequence=["#1D4ED8", "#3B82F6", "#60A5FA","#93C5FD","#BFDBFE"])

        st.plotly_chart(fig, use_container_width=True)






Overwriting Retail_Sales_Analysis.py


In [ ]:
! streamlit run Retail_Sales_Analysis.py

In [5]:
!pipreqs "C:\Users\nayer\Downloads\python course\final project stuff" --encoding=utf-8 --force

INFO: Successfully saved requirements file in C:\Users\nayer\Downloads\python course\final project stuff\requirements.txt
